# Titanic

Dataset-specific starting notebook for the DataFrameSampler paper experiments.

Claim-specific role: small mixed-type benchmark for quick iteration, schema preservation, distributional checks, and downstream classification utility.

## Setup

Run the downloader before executing this notebook:

```bash
python experiments/download_datasets.py
```

In [ ]:
from pathlib import Path
import platform
import time

import numpy as np
import pandas as pd

from dataframe_sampler import ConcreteDataFrameSampler

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
DATA = ROOT / "experiments" / "data" / "processed" / "titanic.csv"
RESULTS = ROOT / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python", platform.python_version())
print("pandas", pd.__version__)
print("dataset", DATA)

In [ ]:
df = pd.read_csv(DATA)
df.shape, df.head()

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(dropna=True),
})
profile

## First DataFrameSampler Run

Titanic is small enough to use directly as a quick mixed-type smoke test.

In [ ]:
vectorizing_columns_dict = {
    "sex": ["age", "fare", "pclass"],
    "class": ["age", "fare", "pclass"],
    "embark_town": ["age", "fare", "pclass"],
    "who": ["age", "fare", "pclass"],
    "adult_male": ["age", "fare", "pclass"],
    "alone": ["age", "fare", "pclass"],
}

sampler = ConcreteDataFrameSampler(
    n_bins=8,
    n_neighbours=6,
    vectorizing_columns_dict=vectorizing_columns_dict,
    embedding_method="pca",
    knn_backend="sklearn",
    random_state=RANDOM_STATE,
)

start = time.perf_counter()
sampler.fit(df)
fit_seconds = time.perf_counter() - start

start = time.perf_counter()
generated = sampler.sample(n_samples=1000)
sample_seconds = time.perf_counter() - start

fit_seconds, sample_seconds, generated.head()

In [ ]:
def quick_similarity_report(real, synthetic):
    numeric = real.select_dtypes(include="number").columns
    categorical = [c for c in real.columns if c not in numeric]
    rows = []
    for column in numeric:
        rows.append({
            "column": column,
            "kind": "numeric",
            "real_mean": real[column].mean(),
            "synthetic_mean": synthetic[column].mean(),
            "abs_mean_delta": abs(real[column].mean() - synthetic[column].mean()),
            "real_missing": real[column].isna().mean(),
            "synthetic_missing": synthetic[column].isna().mean(),
        })
    for column in categorical:
        real_values = set(real[column].dropna().astype(str))
        synthetic_values = set(synthetic[column].dropna().astype(str))
        rows.append({
            "column": column,
            "kind": "categorical",
            "real_unique": len(real_values),
            "synthetic_unique": len(synthetic_values),
            "category_coverage": len(real_values & synthetic_values) / max(len(real_values), 1),
            "real_missing": real[column].isna().mean(),
            "synthetic_missing": synthetic[column].isna().mean(),
        })
    return pd.DataFrame(rows)

report = quick_similarity_report(df, generated)
report

In [ ]:
generated.to_csv(RESULTS / "titanic_generated_start.csv", index=False)
report.to_csv(RESULTS / "titanic_similarity_start.csv", index=False)
pd.DataFrame([
    {"dataset": "titanic", "rows_used": len(df), "generated_rows": len(generated), "fit_seconds": fit_seconds, "sample_seconds": sample_seconds}
]).to_csv(RESULTS / "titanic_runtime_start.csv", index=False)
RESULTS

## Simple Baseline Comparison

Run DataFrameSampler and simple baselines, then summarize them with the TODO point 7 metrics.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

from experiments.compare import run_dataset_comparison

llm_assisted_config = {
    "n_bins": 8,
    "n_neighbours": 6,
    "vectorizing_columns_dict": {
        "sex": ["survived", "pclass", "age", "fare"],
        "class": ["survived", "pclass", "age", "fare"],
        "who": ["survived", "pclass", "age", "fare"],
        "adult_male": ["survived", "pclass", "age", "fare"],
        "deck": ["survived", "pclass", "age", "fare"],
        "embark_town": ["survived", "pclass", "age", "fare"],
        "alive": ["pclass", "age", "fare"],
        "alone": ["survived", "pclass", "age", "fare"],
    },
    "embedding_method": "pca",
    "knn_backend": "sklearn",
    "random_state": RANDOM_STATE,
}

titanic_comparison = run_dataset_comparison(
    df,
    dataset_name="titanic",
    target_column="survived",
    results_dir=RESULTS,
    dataframe_sampler_config={
        "n_bins": 8,
        "n_neighbours": 6,
        "vectorizing_columns_dict": vectorizing_columns_dict,
        "embedding_method": "pca",
        "knn_backend": "sklearn",
        "random_state": RANDOM_STATE,
    },
    llm_assisted_config=llm_assisted_config,
    n_samples=1000,
    random_state=RANDOM_STATE,
)

titanic_comparison